# Task 3 — Text Preprocessing with HuggingFace Transformers

**Goal:** Build a text preprocessing pipeline for our flower captions: clean the
text, tokenize it, and encode it into embeddings using a pretrained HuggingFace
model — producing the exact kind of text representation that Task 5
(attention GAN) and Task 6 (full pipeline) will consume as conditioning input.

**Model choice:** We use **CLIP** (`openai/clip-vit-base-patch32`) rather than a
plain BERT/DistilBERT. CLIP's text encoder is trained jointly with an image
encoder specifically to produce embeddings meaningful for image
generation/matching — exactly our use case in Tasks 5 and 6.

**Design choice (learned from Task 4):** we mount Drive **once**, just to copy
the small `oxford102_metadata.csv` file to local disk. After that single copy,
everything — tokenization, encoding, saving outputs — happens on local disk.
This avoids the slow/unreliable behavior of repeatedly reading over Drive's
network mount.

**Input:** `oxford102_metadata.csv`, produced in Task 4, currently sitting in
your Drive at `elevance-skills/data/oxford102_metadata.csv`.

## 1. Setup

In [ ]:
!pip install -q transformers pandas matplotlib seaborn scikit-learn

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import shutil
from transformers import CLIPTokenizer, CLIPTextModel
from sklearn.metrics.pairwise import cosine_similarity

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 2. Pull the metadata CSV from Drive — once, then work locally

This mounts Drive, copies the one CSV file to `/content/`, and that's the last
time this notebook touches Drive until we optionally save results back at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CSV_PATH = '/content/drive/MyDrive/elevance-skills/data/oxford102_metadata.csv'
LOCAL_CSV_PATH = '/content/oxford102_metadata.csv'

assert os.path.exists(DRIVE_CSV_PATH), (
    f'Could not find {DRIVE_CSV_PATH}. '
    'Double check the path in your Drive matches this, or update DRIVE_CSV_PATH.'
)

shutil.copy(DRIVE_CSV_PATH, LOCAL_CSV_PATH)
print('Copied metadata CSV to local disk:', LOCAL_CSV_PATH)

In [ ]:
df = pd.read_csv(LOCAL_CSV_PATH)
print(f'Loaded {len(df)} rows')
df.head()

## 3. Text cleaning

Our captions are already fairly clean (templated), but we apply standard text
preprocessing steps here — the same steps you'd apply to messier, real-world
caption data like COCO:
- Lowercasing
- Stripping extra whitespace
- Removing stray punctuation (keeping hyphens, since flower names use them)

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)              # collapse multiple spaces
    text = re.sub(r'[^a-z0-9\s\-]', '', text)      # drop stray punctuation, keep hyphens
    return text.strip()

df['clean_caption'] = df['caption'].apply(clean_text)

# sanity check: before vs after
df[['caption', 'clean_caption']].sample(5, random_state=42)

## 4. Tokenization with CLIP's tokenizer

We load `CLIPTokenizer` and inspect what tokenization actually produces:
token IDs, attention masks, and how captions get padded to a fixed length.

In [ ]:
MODEL_NAME = 'openai/clip-vit-base-patch32'

tokenizer = CLIPTokenizer.from_pretrained(MODEL_NAME)

sample_captions = df['clean_caption'].iloc[:4].tolist()
print('Sample captions:')
for c in sample_captions:
    print(' -', c)

encoded = tokenizer(
    sample_captions,
    padding=True,
    truncation=True,
    max_length=32,
    return_tensors='pt'
)

print('\nInput IDs shape:', encoded['input_ids'].shape)
print('Input IDs:\n', encoded['input_ids'])
print('\nAttention mask:\n', encoded['attention_mask'])

In [ ]:
# Decode back to verify tokenization round-trips correctly
for ids in encoded['input_ids']:
    print(tokenizer.decode(ids, skip_special_tokens=True))

## 5. Token length analysis across the full dataset

Understanding the token length distribution helps us choose a sensible
`max_length` for padding/truncation when we batch-encode everything.

In [ ]:
token_lengths = df['clean_caption'].apply(
    lambda t: len(tokenizer(t)['input_ids'])
)

print('Token length stats:')
print(token_lengths.describe())

plt.figure(figsize=(8, 4))
sns.histplot(token_lengths, bins=range(token_lengths.min(), token_lengths.max()+2), color='#4C72B0')
plt.title('Token Length Distribution (CLIP tokenizer)')
plt.xlabel('Number of tokens (incl. special tokens)')
plt.tight_layout()
plt.savefig('/content/token_length_distribution.png', dpi=150)
plt.show()

# choose max_length that comfortably covers ~99th percentile
MAX_LENGTH = int(np.ceil(token_lengths.quantile(0.99)))
print(f'\nChosen MAX_LENGTH = {MAX_LENGTH} (covers 99th percentile of caption lengths)')

## 6. Encoding captions into embeddings with CLIP's text encoder

Tokenizing gives us IDs; now we pass those IDs through the pretrained
`CLIPTextModel` to get actual dense vector embeddings — the representation
that downstream models (Tasks 5 & 6) will condition on.

In [ ]:
text_encoder = CLIPTextModel.from_pretrained(MODEL_NAME).to(device)
text_encoder.eval()

@torch.no_grad()
def encode_captions(captions, batch_size=64):
    """Tokenize + encode a list of captions, returning pooled sentence embeddings."""
    all_embeddings = []
    for i in range(0, len(captions), batch_size):
        batch = captions[i:i+batch_size]
        inputs = tokenizer(
            batch, padding=True, truncation=True,
            max_length=MAX_LENGTH, return_tensors='pt'
        ).to(device)
        outputs = text_encoder(**inputs)
        # pooled_output: the [EOS]-token representation, i.e. a single vector per caption
        pooled = outputs.pooler_output.cpu().numpy()
        all_embeddings.append(pooled)
    return np.concatenate(all_embeddings, axis=0)

# Demo on a small sample first
demo_embeddings = encode_captions(sample_captions)
print('Embedding shape (per caption):', demo_embeddings.shape)

## 7. Sanity check: do similar flowers get similar embeddings?

A good text encoder should place captions about similar/related flowers closer
together in embedding space than unrelated ones. We check this with cosine
similarity on a handful of captions spanning different classes.

In [ ]:
check_captions = [
    'a photo of a rose',
    'a photo of a carnation',        # both cultivated garden flowers -> expect some similarity to rose
    'a photo of a sunflower',
    'a photo of a water lily',
    'a photo of a common dandelion',  # a very different-looking weed flower
]

check_embeddings = encode_captions(check_captions)
sim_matrix = cosine_similarity(check_embeddings)

sim_df = pd.DataFrame(sim_matrix, index=check_captions, columns=check_captions)

plt.figure(figsize=(8, 6))
sns.heatmap(sim_df, annot=True, fmt='.2f', cmap='viridis')
plt.title('Cosine Similarity Between CLIP Text Embeddings')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('/content/caption_similarity_heatmap.png', dpi=150)
plt.show()

## 8. Encode the full dataset (locally, on GPU)

This is the actual deliverable of Task 3: every caption in the dataset,
encoded into a fixed-size embedding vector. Everything so far has stayed on
local disk/GPU memory — fast and reliable.

For ~8,189 captions this runs in well under a minute on a T4 GPU.

In [ ]:
all_captions = df['clean_caption'].tolist()
all_embeddings = encode_captions(all_captions, batch_size=128)

print('Full embeddings matrix shape:', all_embeddings.shape)

LOCAL_EMBEDDINGS_PATH = '/content/caption_embeddings.npy'
LOCAL_CLEAN_CSV_PATH = '/content/oxford102_metadata_clean.csv'

np.save(LOCAL_EMBEDDINGS_PATH, all_embeddings)
df.to_csv(LOCAL_CLEAN_CSV_PATH, index=False)

print('Saved locally:')
print(' -', LOCAL_EMBEDDINGS_PATH)
print(' -', LOCAL_CLEAN_CSV_PATH)
print('Row i of the metadata CSV corresponds to row i of the embeddings array.')

## 9. Copy results to Drive (the ONLY other Drive write in this notebook)

Same pattern as Task 4: two small files copied once, not repeated Drive reads/writes.

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/elevance-skills/data'
os.makedirs(DRIVE_DIR, exist_ok=True)

shutil.copy(LOCAL_EMBEDDINGS_PATH, os.path.join(DRIVE_DIR, 'caption_embeddings.npy'))
shutil.copy(LOCAL_CLEAN_CSV_PATH, os.path.join(DRIVE_DIR, 'oxford102_metadata_clean.csv'))

print('Copied to Drive folder:', DRIVE_DIR)
print(os.listdir(DRIVE_DIR))

## 10. Summary of findings

Fill this in after running the notebook, then copy key points into `NOTES.md`
and today's daily log. Things to note:
- Chosen `MAX_LENGTH` and why
- Shape/dimensionality of the final embeddings
- What the similarity heatmap showed (did related flowers cluster more closely?)
- Any surprises in tokenization behavior